In [0]:
spark.table("ecommerce_dev.silver.geolocation").printSchema()
spark.table("ecommerce_dev.silver.geolocation").show(5, truncate=False)

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- has_invalid_coordinates: boolean (nullable = true)

+---------------------------+-------------------+-------------------+----------------+-----------------+-----------------------+-----------------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng    |geolocation_city|geolocation_state|bronze_ingested_at     |has_invalid_coordinates|
+---------------------------+-------------------+-------------------+----------------+-----------------+-----------------------+-----------------------+
|01009                      |-23.545429533441073|-46.63571531432852 |sao paulo       |SP               |2026-08-04 00:55:03.567|false                  |
|01040                 

In [0]:
spark.sql("""
  SELECT geolocation_zip_code_prefix, COUNT(*) AS cnt
  FROM ecommerce_dev.silver.geolocation
  GROUP BY geolocation_zip_code_prefix
  ORDER BY cnt DESC
  LIMIT 10
""").show()

+---------------------------+---+
|geolocation_zip_code_prefix|cnt|
+---------------------------+---+
|                      38400|779|
|                      35500|751|
|                      11680|727|
|                      11740|678|
|                      36400|627|
|                      38408|621|
|                      39400|620|
|                      35162|611|
|                      37200|596|
|                      35900|589|
+---------------------------+---+



In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.dim_geolocation (
    geolocation_key BIGINT GENERATED ALWAYS AS IDENTITY,
    zip_code_prefix STRING NOT NULL,
    city STRING,
    state STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    has_invalid_coordinates BOOLEAN,
    gold_updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Gold geolocation dimension. Grain: one row per zip_code_prefix, aggregated from crowdsourced Silver rows. Coordinates are the average of valid rows; city/state is the most frequent value per prefix.'
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

geo_silver = spark.table("ecommerce_dev.silver.geolocation")

# Average coordinates from valid rows only
coords_agg = (
    geo_silver
    .filter(col("has_invalid_coordinates") == False)
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("latitude"),
        avg("geolocation_lng").alias("longitude")
    )
)

# Most frequent city/state per prefix
city_state_counts = (
    geo_silver
    .groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(count("*").alias("cnt"))
)
window_spec = Window.partitionBy("geolocation_zip_code_prefix").orderBy(col("cnt").desc())
mode_city_state = (
    city_state_counts
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .select(
        col("geolocation_zip_code_prefix"),
        col("geolocation_city").alias("city"),
        col("geolocation_state").alias("state")
    )
)

# Flag: prefixes with zero valid coordinate rows
valid_prefixes = coords_agg.select("geolocation_zip_code_prefix").distinct()
all_prefixes = geo_silver.select("geolocation_zip_code_prefix").distinct()
invalid_only_prefixes = all_prefixes.subtract(valid_prefixes)

stg_geolocation = (
    mode_city_state
    .join(coords_agg, "geolocation_zip_code_prefix", "left")
    .withColumn("has_invalid_coordinates",
                col("geolocation_zip_code_prefix").isin(
                    [r["geolocation_zip_code_prefix"] for r in invalid_only_prefixes.collect()]
                ))
    .select(
        col("geolocation_zip_code_prefix").alias("zip_code_prefix"),
        "city", "state", "latitude", "longitude", "has_invalid_coordinates"
    )
)

stg_geolocation.createOrReplaceTempView("stg_geolocation")

In [0]:
total = stg_geolocation.count()
nulls = stg_geolocation.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in ["latitude", "longitude", "city", "state"]]
)
print(f"Row count: {total}")
nulls.show()

Row count: 19015
+--------+---------+----+-----+
|latitude|longitude|city|state|
+--------+---------+----+-----+
|       5|        5|   0|    0|
+--------+---------+----+-----+



In [0]:
stg_geolocation.filter(col("latitude").isNull()).show(10, truncate=False)

+---------------+-----------------------+-----+--------+---------+-----------------------+
|zip_code_prefix|city                   |state|latitude|longitude|has_invalid_coordinates|
+---------------+-----------------------+-----+--------+---------+-----------------------+
|18243          |bom retiro da esperanca|SP   |NULL    |NULL     |true                   |
|53990          |fernando de noronha    |PE   |NULL    |NULL     |true                   |
|78131          |varzea grande          |MT   |NULL    |NULL     |true                   |
|83252          |ilha dos valadares     |PR   |NULL    |NULL     |true                   |
|95130          |santa lucia do piai    |RS   |NULL    |NULL     |true                   |
+---------------+-----------------------+-----+--------+---------+-----------------------+



In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "ecommerce_dev.gold.dim_geolocation")

(target.alias("t")
 .merge(
     stg_geolocation.alias("s"),
     "t.zip_code_prefix = s.zip_code_prefix"
 )
 .whenMatchedUpdate(set={
     "city": "s.city",
     "state": "s.state",
     "latitude": "s.latitude",
     "longitude": "s.longitude",
     "has_invalid_coordinates": "s.has_invalid_coordinates",
     "gold_updated_at": "current_timestamp()"
 })
 .whenNotMatchedInsert(values={
     "zip_code_prefix": "s.zip_code_prefix",
     "city": "s.city",
     "state": "s.state",
     "latitude": "s.latitude",
     "longitude": "s.longitude",
     "has_invalid_coordinates": "s.has_invalid_coordinates",
     "gold_updated_at": "current_timestamp()"
 })
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("ALTER TABLE ecommerce_dev.gold.dim_geolocation ALTER COLUMN geolocation_key SET NOT NULL")
spark.sql("ALTER TABLE ecommerce_dev.gold.dim_geolocation ADD CONSTRAINT pk_dim_geolocation PRIMARY KEY (geolocation_key)")

DataFrame[]

In [0]:
spark.sql("""
  SELECT COUNT(*) AS total,
         SUM(CASE WHEN has_invalid_coordinates THEN 1 ELSE 0 END) AS invalid_coord_count
  FROM ecommerce_dev.gold.dim_geolocation
""").show()

+-----+-------------------+
|total|invalid_coord_count|
+-----+-------------------+
|19015|                  5|
+-----+-------------------+



In [0]:
print("SILVER TABLES:")
spark.sql("SHOW TABLES IN ecommerce_dev.silver").show(truncate=False)

print("GOLD TABLES:")
spark.sql("SHOW TABLES IN ecommerce_dev.gold").show(truncate=False)

SILVER TABLES:
+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|silver  |category_translation|false      |
|silver  |customers           |false      |
|silver  |geolocation         |false      |
|silver  |order_items         |false      |
|silver  |order_payments      |false      |
|silver  |order_reviews       |false      |
|silver  |orders              |false      |
|silver  |products            |false      |
|silver  |sellers             |false      |
|        |stg_geolocation     |true       |
+--------+--------------------+-----------+

GOLD TABLES:
+--------+----------------+-----------+
|database|tableName       |isTemporary|
+--------+----------------+-----------+
|gold    |dim_customer    |false      |
|gold    |dim_date        |false      |
|gold    |dim_geolocation |false      |
|gold    |dim_product     |false      |
|gold    |dim_seller      |false      |
|gold    |fact_order_items|false   